In [10]:
import json
import os

import pandas as pd
from tqdm import tqdm
from collections import defaultdict
from openai import OpenAI
from dotenv import load_dotenv

from utils.data import edinet_to_industry_map
from utils.datetime import date_string_to_quarter
from utils.edinet_api import get_doc_name
from utils.ELO import EloRatingSystem, get_num_games, generate_random_pdf_pair
from utils.signal import get_winner, get_stock_code_from_path

from utils.datapath import (
    documents_path,
    edinet_codes_path,
    docs_metadata_path,
    industry_elo_signals_path
)

In [2]:
load_dotenv()
XAI_API_KEY = os.getenv("XAI_API_KEY")

client = OpenAI(
    api_key=XAI_API_KEY,
    base_url="https://api.x.ai/v1",
)

In [3]:
df = pd.read_excel(edinet_codes_path)

with open(docs_metadata_path) as f:
    docs_metadata = json.load(f)

In [4]:
industry_quarterly_docs = defaultdict(lambda: defaultdict(list))

for doc in docs_metadata:
    industry = edinet_to_industry_map.get(doc['edinetCode'], "na")
    period_end_quater = date_string_to_quarter(doc["periodEnd"])

    industry_path = os.path.join(documents_path, industry)
    quarter_path = os.path.join(industry_path, period_end_quater)

    save_name = get_doc_name(doc)
    output_path = os.path.join(quarter_path, save_name)

    industry_quarterly_docs[industry][period_end_quater].append(output_path)

In [5]:
industry_quarterly_signals = {}

for industry, quarterly_docs in tqdm(industry_quarterly_docs.items()):
    elo_system = EloRatingSystem(ratings={})
    quarterly_signals = {}

    for quarter, docs in quarterly_docs.items():
        for pdf1_path, pdf2_path in generate_random_pdf_pair(
            docs, get_num_games(len(docs))
        ):
            winner = get_winner(pdf1_path, pdf2_path)
            companyA_code, companyB_code = [
                get_stock_code_from_path(path) for path in (pdf1_path, pdf2_path)
            ]
            elo_system.update_ratings(companyA_code, companyB_code, winner)

        quarterly_signals[quarter] = elo_system.get_all_ratings()
    
    industry_quarterly_signals[industry] = quarterly_signals


  0%|          | 0/33 [00:00<?, ?it/s]

100%|██████████| 33/33 [00:30<00:00,  1.09it/s]


In [6]:
# quarterly_signals should equal industry_quarterly_signals[industry]
# industry_quarterly_signals["Marine Transportation"]

In [ ]:
# Industry wise elo - DONE
# Industry wise portfolio backtest

# But that's not the important point
# Get to the api calls

In [11]:
# Industry wise elo

with open(industry_elo_signals_path, 'w', encoding='utf-8') as f:
    json.dump(industry_quarterly_signals, f)